# Extraction de données multimodales — Google Fact Check Tools API

Ce notebook interroge l'API **Claims Search** de Google Fact Check Tools pour collecter des affirmations vérifiées (fact-checks), puis enrichit chaque résultat avec l'image associée à l'article de vérification, afin de constituer un jeu de données **multimodal (texte + image)**.

## Sommaire
1. Configuration
2. Client API — recherche de claims
3. Collecte paginée sur plusieurs requêtes
4. Mise en dataframe
5. Enrichissement multimodal (extraction d'image par page)
6. Téléchargement des images
7. Sauvegarde du dataset consolidé
8. Exploration rapide

## Prérequis
- Une clé API Google Cloud avec la **Fact Check Tools API** activée.
- La clé placée dans un fichier `.env` à la racine du projet : `GOOGLE_FACTCHECK_API_KEY=...`

## 1. Configuration

In [ ]:
import os
import time
import json
from pathlib import Path
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.notebook import tqdm

In [ ]:
load_dotenv()

API_KEY = os.environ.get("GOOGLE_FACTCHECK_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing GOOGLE_FACTCHECK_API_KEY. Set it in a .env file at the project root."
    )

CLAIMS_SEARCH_URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"

# Output locations
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "google_factcheck"
IMAGES_DIR = DATA_DIR / "images"
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Be polite with external services
REQUEST_DELAY_SECONDS = 1.0
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Projet12-FactCheckCollector/1.0)"}

# Only keep fact-checks published within this many days: older reviews are more likely
# to point to dead/invalid links
MAX_AGE_DAYS = 180

## 2. Client API — recherche de claims

Wrapper autour de l'endpoint `claims:search`, avec gestion des erreurs HTTP et du throttling.

In [ ]:
def search_claims(
    query: str | None = None,
    language_code: str | None = None,
    page_size: int = 1,
    page_token: str | None = None,
    max_age_days: int | None = None,
    review_publisher_site_filter: str | None = None,
) -> dict:
    """Call the Google Fact Check Tools claims:search endpoint once.

    Either query or review_publisher_site_filter must be provided (the API requires one).
    language_code left unset (None) searches all languages.
    """
    if not query and not review_publisher_site_filter:
        raise ValueError("Provide at least one of query or review_publisher_site_filter")

    params = {
        "key": API_KEY,
        "pageSize": page_size,
    }
    if query:
        params["query"] = query
    if language_code:
        params["languageCode"] = language_code
    if page_token:
        params["pageToken"] = page_token
    if max_age_days:
        params["maxAgeDays"] = max_age_days
    if review_publisher_site_filter:
        params["reviewPublisherSiteFilter"] = review_publisher_site_filter

    response = requests.get(CLAIMS_SEARCH_URL, params=params, timeout=30)
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        # Surface Google's actual error body (raise_for_status only gives the status code)
        raise requests.HTTPError(f"{error}\nResponse body: {response.text}") from error
    return response.json()

## 3. Collecte paginée

On suit les `nextPageToken` jusqu'à épuisement ou jusqu'à une limite de résultats, en filtrant sur `maxAgeDays` pour ne récupérer que des fact-checks récents (les liens plus anciens sont souvent morts).

In [ ]:
def collect_claims(
    query: str | None = None,
    language_code: str | None = None,
    max_results: int = 300,
    page_size: int = 20,
    max_age_days: int | None = MAX_AGE_DAYS,
    review_publisher_site_filter: str | None = None,
) -> list[dict]:
    """Collect raw claim records for a single query and/or publisher site, following pagination."""
    all_claims: list[dict] = []
    page_token = None

    with tqdm(total=max_results, desc=review_publisher_site_filter or query) as progress_bar:
        while True:
            payload = search_claims(
                query=query,
                language_code=language_code,
                page_size=page_size,
                page_token=page_token,
                max_age_days=max_age_days,
                review_publisher_site_filter=review_publisher_site_filter,
            )
            claims = payload.get("claims", [])
            all_claims.extend(claims)
            progress_bar.update(len(claims))

            page_token = payload.get("nextPageToken")
            time.sleep(REQUEST_DELAY_SECONDS)

            if not page_token or len(all_claims) >= max_results or not claims:
                break

    return all_claims

## 4. Mise en dataframe

Chaque claim peut contenir plusieurs `claimReview` (une par média de vérification) : on met les résultats à plat, une ligne par couple claim/review.

In [ ]:
def claims_to_dataframe(raw_claims: list[dict]) -> pd.DataFrame:
    """Flatten raw claims:search results into one row per claim review."""
    rows = []
    for claim in raw_claims:
        for review in claim.get("claimReview", []):
            publisher = review.get("publisher", {})
            rows.append(
                {
                    "claim_text": claim.get("text"),
                    "claimant": claim.get("claimant"),
                    "claim_date": claim.get("claimDate"),
                    "review_publisher_name": publisher.get("name"),
                    "review_publisher_site": publisher.get("site"),
                    "review_url": review.get("url"),
                    "review_title": review.get("title"),
                    "review_date": review.get("reviewDate"),
                    "textual_rating": review.get("textualRating"),
                    "language_code": review.get("languageCode"),
                }
            )
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["review_url"]).reset_index(drop=True)
    return df

## 5. Exécution de la collecte

Filtrer par mot-clé (`query`) restreint trop le volume — on interroge à la place directement une liste de fact-checkers connus via `reviewPublisherSiteFilter`, sans aucun filtre thématique, puis on fusionne et déduplique les résultats.

In [ ]:
FACT_CHECK_SITES = [
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "leadstories.com",
    "factcheck.afp.com",
    "reuters.com",
    "apnews.com",
    "africacheck.org",
    "boomlive.in",
    "altnews.in",
    "correctiv.org",
    "maldita.es",
    "newtral.es",
    "rappler.com",
    "verafiles.org",
]
PER_SITE_MAX_RESULTS = 200

raw_claims = []
for site in FACT_CHECK_SITES:
    raw_claims.extend(
        collect_claims(
            review_publisher_site_filter=site,
            max_results=PER_SITE_MAX_RESULTS,
            max_age_days=MAX_AGE_DAYS,
        )
    )
print(f"Collected {len(raw_claims)} raw claims across {len(FACT_CHECK_SITES)} publishers")

df = claims_to_dataframe(raw_claims)
print(f"Flattened into {len(df)} unique claim reviews")
df.head()

## 5b. Filtrage par date

`maxAgeDays` filtre déjà côté API, mais `reviewDate` n'est pas toujours renseigné ou fiable : on applique un second filtrage explicite ici, et on écarte les lignes sans date exploitable.

In [ ]:
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce", utc=True)

cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=MAX_AGE_DAYS)
before = len(df)
df = df[df["review_date"].notna() & (df["review_date"] >= cutoff)].reset_index(drop=True)

print(f"Kept {len(df)}/{before} rows published after {cutoff.date()}")
df.head()

## 5c. Normalisation du label (fake / true)

`textual_rating` est un texte libre choisi par chaque organisme de fact-checking ("Pants on Fire", "Mostly True", "Faux", "Trompeur", ...). Pour entraîner un modèle binaire, on le normalise en un label clair `{"fake", "true"}` à partir de mots-clés, et on **rejette** les évaluations ambiguës (mi-vrai, satire, non vérifiable, ...) plutôt que de les forcer dans une catégorie.

Le déséquilibre observé (très peu de `"true"`) reflète le biais naturel des fact-checkers, qui vérifient surtout des affirmations fausses. On y répond en élargissant la collecte (`max_results` plus grand ci-dessus) plutôt qu'en filtrant davantage ; un rééquilibrage optionnel (sous-échantillonnage de la classe majoritaire) est proposé en fin de section."

In [ ]:
import re

# Ratings that are inherently ambiguous for a binary fake/true task: excluded rather than guessed
AMBIGUOUS_RATING_PATTERNS = [
    "half true", "half false", "mixture", "mixed", "unproven", "unverified",
    "opinion", "satire", "outdated", "missing context", "out of context",
    "needs context", "unclear", "research in progress", "altered context",
    "mi-vrai", "mi-faux", "à vérifier", "sans fondement suffisant",
]
FAKE_RATING_PATTERNS = [
    "false", "fake", "pants on fire", "misleading", "incorrect", "not true",
    "no evidence", "no such report", "distort", "manipulated", "hoax", "scam",
    "alter", "miscaption",
    "faux", "trompeur", "mensonge", "inexact", "erroné", "erronee",
]
TRUE_RATING_PATTERNS = [
    "true", "correct", "accurate", "vrai", "exact", "verified", "confirmed",
]


def normalize_rating(textual_rating: str | float) -> str | None:
    """Map a free-text textual_rating to a clean binary label, or None if ambiguous/unrecognized."""
    if pd.isna(textual_rating):
        return None
    rating = textual_rating.strip().lower()

    if any(re.search(pattern, rating) for pattern in AMBIGUOUS_RATING_PATTERNS):
        return None
    if any(re.search(pattern, rating) for pattern in FAKE_RATING_PATTERNS):
        return "fake"
    if any(re.search(pattern, rating) for pattern in TRUE_RATING_PATTERNS):
        return "true"
    return None

In [ ]:
df["label"] = df["textual_rating"].apply(normalize_rating)

before = len(df)
unmapped_ratings = df.loc[df["label"].isna(), "textual_rating"].value_counts()
df = df[df["label"].notna()].reset_index(drop=True)

print(f"Kept {len(df)}/{before} rows with a clear fake/true label")
print("\nClass balance:")
print(df["label"].value_counts())
print("\nTop unmapped ratings (review AMBIGUOUS/FAKE/TRUE_RATING_PATTERNS if this list looks wrong):")
print(unmapped_ratings.head(15))

### Rééquilibrage optionnel

Si la classe minoritaire reste trop petite après la collecte élargie, augmenter `max_results` (section 5) est préférable à un sous-échantillonnage agressif qui jetterait la majorité des données. Ce sous-échantillonnage n'est donc appliqué que si la classe minoritaire compte un nombre raisonnable d'exemples (seuil `MIN_MINORITY_CLASS_SIZE` ci-dessous) ; sinon, il est ignoré avec un avertissement."

In [ ]:
MIN_MINORITY_CLASS_SIZE = 30

class_counts = df["label"].value_counts()
minority_label, minority_count = class_counts.idxmin(), class_counts.min()

if minority_count < MIN_MINORITY_CLASS_SIZE:
    print(
        f"Minority class '{minority_label}' has only {minority_count} rows: "
        f"skipping rebalancing, increase max_results in section 5 instead."
    )
else:
    df = (
        df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(n=minority_count, random_state=42))
        .reset_index(drop=True)
    )
    print(f"Rebalanced to {minority_count} rows per class:")
    print(df["label"].value_counts())

## 6. Enrichissement multimodal — extraction d'image par page

L'API Claims Search ne fournit pas d'image directement. On récupère l'URL de l'image principale (`og:image` / `twitter:image`) de chaque page de vérification pour obtenir un couple texte + image exploitable.

In [ ]:
def fetch_main_image_url(page_url: str) -> str | None:
    """Fetch a review article page and extract its main image URL from meta tags."""
    try:
        response = requests.get(page_url, headers=HTTP_HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException:
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    for meta_name in ("og:image", "twitter:image", "twitter:image:src"):
        tag = soup.find("meta", attrs={"property": meta_name}) or soup.find(
            "meta", attrs={"name": meta_name}
        )
        if tag and tag.get("content"):
            return urljoin(page_url, tag["content"])
    return None

In [ ]:
image_urls = []
for review_url in tqdm(df["review_url"], desc="Fetching images"):
    image_urls.append(fetch_main_image_url(review_url))
    time.sleep(REQUEST_DELAY_SECONDS)

df["image_url"] = image_urls
df["image_url"].notna().mean()

## 7. Téléchargement des images

In [ ]:
def download_image(image_url: str, dest_path: Path) -> bool:
    """Download a single image to dest_path. Returns True on success."""
    try:
        response = requests.get(image_url, headers=HTTP_HEADERS, timeout=15)
        response.raise_for_status()
        dest_path.write_bytes(response.content)
        return True
    except requests.RequestException:
        return False

In [ ]:
local_paths = []
for idx, image_url in tqdm(list(enumerate(df["image_url"])), desc="Downloading images"):
    if pd.isna(image_url):
        local_paths.append(None)
        continue

    extension = Path(image_url.split("?")[0]).suffix or ".jpg"
    dest_path = IMAGES_DIR / f"claim_{idx:05d}{extension}"

    if download_image(image_url, dest_path):
        local_paths.append(str(dest_path.relative_to(PROJECT_ROOT)))
    else:
        local_paths.append(None)
    time.sleep(REQUEST_DELAY_SECONDS)

df["local_image_path"] = local_paths
df["local_image_path"].notna().sum()

## 8. Sauvegarde du dataset consolidé

In [ ]:
output_csv = DATA_DIR / "google_factcheck_multimodal.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} rows to {output_csv}")

## 9. Exploration rapide

In [ ]:
df.info()

In [ ]:
df["label"].value_counts()

In [ ]:
df[df["local_image_path"].notna()][
    ["claim_text", "textual_rating", "review_publisher_name", "local_image_path"]
].head(10)